# 餐饮客户满意度预测

In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier
import joblib

## 1. 获取数据，检查并处理缺失值和重复值

In [28]:
df = pd.read_csv('数据/餐饮客户满意度.csv')
print(df.info())
print('数据集形状:', df.shape)
print('缺失值数量:', df.isnull().sum().sum())
print('重复值数量:', df.duplicated().sum())
df = df.drop_duplicates()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   客户ID    25 non-null     int64
 1   就餐次数    25 non-null     int64
 2   菜品口味评分  25 non-null     int64
 3   服务态度评分  25 non-null     int64
 4   餐厅环境评分  25 non-null     int64
 5   等待时间    25 non-null     int64
 6   人均消费    25 non-null     int64
 7   是否会员    25 non-null     str  
 8   是否满意    25 non-null     str  
dtypes: int64(7), str(2)
memory usage: 1.9 KB
None
数据集形状: (25, 9)
缺失值数量: 0
重复值数量: 0


,客户ID,就餐次数,菜品口味评分,服务态度评分,餐厅环境评分,等待时间,人均消费,是否会员,是否满意
0,1,3,4,5,4,15,80,是,是
1,2,1,3,4,3,5,50,否,否
2,3,5,5,5,5,0,120,是,是
3,4,2,4,4,4,10,70,是,是
4,5,4,3,3,3,20,90,否,否


## 2. 对分类特征进行编码

In [17]:
le_member = LabelEncoder()
le_satisfaction = LabelEncoder()

df['是否会员'] = le_member.fit_transform(df['是否会员'])
df['是否满意'] = le_satisfaction.fit_transform(df['是否满意'])

print('是否会员编码:', dict(zip(le_member.classes_, le_member.transform(le_member.classes_))))
print('是否满意编码:', dict(zip(le_satisfaction.classes_, le_satisfaction.transform(le_satisfaction.classes_))))
df.head()

是否会员编码: {'否': np.int64(0), '是': np.int64(1)}
是否满意编码: {'否': np.int64(0), '是': np.int64(1)}


,客户ID,就餐次数,菜品口味评分,服务态度评分,餐厅环境评分,等待时间,人均消费,是否会员,是否满意
0,1,3,4,5,4,15,80,1,1
1,2,1,3,4,3,5,50,0,0
2,3,5,5,5,5,0,120,1,1
3,4,2,4,4,4,10,70,1,1
4,5,4,3,3,3,20,90,0,0


## 3. 特征选择，划分特征和标签

In [18]:
X = df.drop(['客户ID', '是否满意'], axis=1)
y = df['是否满意']
print('特征列:', X.columns.tolist())
print('特征形状:', X.shape)
print('标签形状:', y.shape)

特征列: ['就餐次数', '菜品口味评分', '服务态度评分', '餐厅环境评分', '等待时间', '人均消费', '是否会员']
特征形状: (25, 7)
标签形状: (25,)


## 4. 划分训练集和测试集（7:3）

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print('训练集大小:', X_train.shape[0])
print('测试集大小:', X_test.shape[0])

训练集大小: 17
测试集大小: 8


## 5. XGBoost和GBDT建模

In [20]:
xgb_model = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)
print('XGBoost模型训练完成')

gbdt_model = GradientBoostingClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
gbdt_model.fit(X_train, y_train)
print('GBDT模型训练完成')

XGBoost模型训练完成
GBDT模型训练完成


## 6. 网格搜索调参

In [21]:
xgb_params = {
    'n_estimators': [50, 100], 
    'max_depth': [3, 4, 5],
    'learning_rate': [0.05, 0.1, 0.2]}

xgb_grid = GridSearchCV(
    XGBClassifier(andom_state=42,eval_metric='logloss'), 
    xgb_params, 
    cv=3, 
    scoring='accuracy')

xgb_grid.fit(X_train, y_train)

print('XGBoost最优参数:', xgb_grid.best_params_)
print('XGBoost最优分数:', xgb_grid.best_score_)

XGBoost最优参数: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 50}
XGBoost最优分数: 0.8333333333333334


In [22]:
gbdt_params = {
    'n_estimators': [50, 100],
    'max_depth': [3, 4, 5], 
    'learning_rate': [0.05, 0.1, 0.2]}

gbdt_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42), 
    gbdt_params, 
    cv=3, 
    scoring='accuracy')

gbdt_grid.fit(X_train, y_train)

print('GBDT最优参数:', gbdt_grid.best_params_)
print('GBDT最优分数:', gbdt_grid.best_score_)

GBDT最优参数: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 50}
GBDT最优分数: 0.8333333333333334


## 7. 模型性能评估

In [29]:
y_pred = xgb_model.predict(X_test)
print('XGBoost默认:')
print(f'  准确率: {accuracy_score(y_test, y_pred)}')
print(f'  精确率: {precision_score(y_test, y_pred)}')
print(f'  召回率: {recall_score(y_test, y_pred)}')
print(f'  F1分数: {f1_score(y_test, y_pred)}')
f1_xgb = f1_score(y_test, y_pred)

XGBoost默认:
  准确率: 1.0
  精确率: 1.0
  召回率: 1.0
  F1分数: 1.0


In [24]:
y_pred = gbdt_model.predict(X_test)
print('GBDT默认:')
print(f'  准确率: {accuracy_score(y_test, y_pred)}')
print(f'  精确率: {precision_score(y_test, y_pred)}')
print(f'  召回率: {recall_score(y_test, y_pred)}')
print(f'  F1分数: {f1_score(y_test, y_pred)}')
f1_gbdt = f1_score(y_test, y_pred)

GBDT默认:
  准确率: 1.0
  精确率: 1.0
  召回率: 1.0
  F1分数: 1.0


In [25]:
y_pred = xgb_grid.best_estimator_.predict(X_test)
print('XGBoost调参:')
print(f'  准确率: {accuracy_score(y_test, y_pred)}')
print(f'  精确率: {precision_score(y_test, y_pred)}')
print(f'  召回率: {recall_score(y_test, y_pred)}')
print(f'  F1分数: {f1_score(y_test, y_pred)}')
f1_xgb_grid = f1_score(y_test, y_pred)

XGBoost调参:
  准确率: 1.0
  精确率: 1.0
  召回率: 1.0
  F1分数: 1.0


In [26]:
y_pred = gbdt_grid.best_estimator_.predict(X_test)
print('GBDT调参:')
print(f'  准确率: {accuracy_score(y_test, y_pred)}')
print(f'  精确率: {precision_score(y_test, y_pred)}')
print(f'  召回率: {recall_score(y_test, y_pred)}')
print(f'  F1分数: {f1_score(y_test, y_pred)}')
f1_gbdt_grid = f1_score(y_test, y_pred)

GBDT调参:
  准确率: 1.0
  精确率: 1.0
  召回率: 1.0
  F1分数: 1.0


## 8. 选择最优模型并保存

In [27]:
models = [
    [xgb_model, 'XGBoost默认', f1_xgb],
    [gbdt_model, 'GBDT默认', f1_gbdt],
    [xgb_grid.best_estimator_, 'XGBoost调参', f1_xgb_grid],
    [gbdt_grid.best_estimator_, 'GBDT调参', f1_gbdt_grid]
]
best_model, best_name, best_f1 = max(models, key=lambda x: x[2])

print('最优模型:', best_name)
print('最优F1分数:', best_f1)

joblib.dump(best_model, 'best_satisfaction_model.pkl')
print('最优模型已保存')

最优模型: XGBoost默认
最优F1分数: 1.0
最优模型已保存
